# ACASXU Workshop Tutorial

This notebook shows how to load an ACASXU benchmark network in StarV, build the corresponding input Star set, and read the VNNLIB property used for verification.


## 1. Setup

Import the StarV utilities used in the workshop and resolve paths relative to the local StarV package. This keeps the notebook working even when Jupyter is launched from a different folder.


In [1]:
# Numerical arrays are used for bounds and parsed VNNLIB data.
import numpy as np
import StarV

# Star is the symbolic set representation used by StarV reachability routines.
from StarV.set.star import Star

# load_ACASXU loads the curated ACASXU benchmark from StarV MAT files.
# load_neural_network demonstrates loading a network file such as ONNX.
from StarV.util.load import load_ACASXU, load_neural_network

# VNNLIB helpers read benchmark input boxes and output safety constraints.
from StarV.util.vnnlib import read_vnnlib_simple, get_num_inputs_outputs
from StarV.util.lp_solver import sample

from pathlib import Path

# repo_root points at the repository root that contains the StarV package folder.
repo_root = Path(StarV.__file__).resolve().parents[1]

## 2. Load The Built-In ACASXU Benchmark

For most StarV ACASXU examples, load_ACASXU is the simplest path. It returns the network, normalized input bounds, and the unsafe output region for the selected property.


In [9]:
# ACASXU networks are indexed by (x, y). This workshop uses network 1_1.
net_id = (1, 1)

# spec_id selects the ACASXU safety property. Property 1 constrains the COC score.
spec_id = 1

# The loader returns both the network and the verification property data.
net, lb, ub, unsafe_mat, unsafe_vec = load_ACASXU(*net_id, spec_id)
print('Information about the loaded ACASXU network and property:')
print(net)

# lb and ub are already normalized using the ACASXU scaling constants.
print(f"Input Specification for ACASXU property {spec_id}:")
print(f"lb: {lb}")
print(f"ub: {ub}")

# A Star set compactly represents all inputs in the box [lb, ub].
print("Constructing a Star set from the input bounds...")
S = Star(lb, ub)

print(f"Star: {S}")
repr(S)

print(f"Unsafe output constraints (unsafe_mat, unsafe_vec):")
print(f"{unsafe_mat} <= {unsafe_vec}")

Information about the loaded ACASXU network and property:

=============NETWORK===============
Network type: ffnn_ACASXU_1_1
Input Dimension: 5
Output Dimension: 5
Number of Layers: 13
Layer types:
Layer 0: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 5, dtype=float64)
Layer 1: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 2: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 3: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 4: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 5: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 6: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 7: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 8: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 9: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 10: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (

## 3. Load The ONNX Network And VNNLIB Property

This section mirrors a standard verification benchmark layout: an ONNX network plus a VNNLIB property file. The VNNLIB parser provides the input box and output constraints.


In [ ]:
# ACASXU networks are indexed by (x, y). This workshop uses network 1_1.
net_id = (1, 1)

# spec_id selects the ACASXU safety property. Property 1 constrains the COC score.
spec_id = 1

# Path to the ONNX copy of ACASXU 1_1.
network_path = str(repo_root / f"StarV/util/data/nets/ACASXU/onnx/ACASXU_run2a__{net_id[0]}_{net_id[1]}_batch_2000.onnx")
net_type = "ACASXU_FFNN_1_1"

print('Information about the loaded ACASXU network and property:')
# Convert the ONNX network into StarV layers.
net = load_neural_network(network_path, net_type=net_type)
print(net)

# Property 1 is stored in VNNLIB format next to the ONNX benchmark files.
vnnlib_file_dir = str(repo_root / f"StarV/util/data/nets/ACASXU/onnx/prop_{spec_id}.vnnlib")

# The parser needs the ONNX input/output dimensions to interpret the property.
num_inputs, num_outputs, inp_dtype = get_num_inputs_outputs(network_path)
vnnlib_rv = read_vnnlib_simple(vnnlib_file_dir, num_inputs, num_outputs)

# Each VNNLIB entry is an input box plus a list of output constraints.
box, spec_list = vnnlib_rv[0]
bounds = np.array(box, dtype=inp_dtype)
unsafe_mat, unsafe_vec = spec_list[0]

# Convert the parsed input box into lower and upper bound vectors.
lb, ub = bounds[:, 0], bounds[:, 1]
print(f"Input Specification for ACASXU property {spec_id}:")
print(f"lb: {lb}")
print(f"ub: {ub}")

# Build the corresponding Star input set for reachability.
print("Constructing a Star set from the input bounds...")
S = Star(lb, ub)

print(f"Star: {S}")
repr(S)

print(f"Unsafe output constraints (unsafe_mat, unsafe_vec):")
print(f"{unsafe_mat} <= {unsafe_vec}")

Information about the loaded ACASXU network and property:

=============NETWORK===============
Network type: ACASXU_FFNN_1_1
Input Dimension: 5
Output Dimension: 5
Number of Layers: 13
Layer types:
Layer 0: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 5, dtype=float64)
Layer 1: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 2: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 3: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 4: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 5: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 6: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 7: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 8: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (50, 50, dtype=float64)
Layer 9: <class 'StarV.layer.ReLULayer.ReLULayer'>
Layer 10: <class 'StarV.layer.FullyConnectedLayer.FullyConnectedLayer'> (

In [7]:
spec_list

[(array([[-1.,  0.,  0.,  0.,  0.]]), array([-3.99113]))]

## 4. Workshop Exercise Space

Use the empty cell below to try a different ACASXU network, change the property id, or inspect variables such as unsafe_mat, unsafe_vec, spec_list, and vnnlib_rv.


In [8]:
unsafe_mat

array([-1,  0,  0,  0,  0])